Integrantes:
*   Cortés Sanginez Jorge Orlando
*   Gascón Calixto Ruben
*   Gonzalez Díaz Garzón Gerardo Felipe
*   Hernández Mejía Geovanna
*   Salmoran Acuña Jonathan Ivan

# Práctica 1: Simulación de Cartera de Responsabilidad Civil Riesgos Profesionales – Datos SESA 2024

En este cuaderno construiremos un dataset sintético para representar una cartera de pólizas de Responsabilidad Civil en Riesgos profesionales. Asumimos cinco coberturas principales: **Actividades e inmuebles (AeI), RC de Directores y Ejecutivos de alto nivel (DO), Errores y Omisiones (EO), RC Hospitales (RChosp) y RC Profesionistas (RCprof).** Los parámetros de frecuencia y severidad se basan en supuestos ficticios que pueden ser ajustados.

### Importación de librerías y semilla

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fijamos semilla para reproducibilidad
np.random.seed(2025)


```
# freq_base = {"AeI": 0.0032796, "DO": 0.0000001, "EO": 0.0003255, "RChosp": 0.0001202, "RCprof": 0.0128781}
```



### Definimos parámetros y segmentos segun lo obtenido en SESA

In [2]:
# Número de pólizas a simular
n_polizas = 199719

# Frecuencias base por cobertura (probabilidad de siniestro por año)
freq_base = {"AeI": 0.0032796, "DO": 0.0000001, "EO": 0.0003255, "RChosp": 0.0001202, "RCprof": 0.0128781}

# Severidades medias por cobertura (en pesos)
sev_media = {"AeI": 5118, "DO": 273299, "EO": 3157, "RChosp": 28543, "RCprof": 61486}

# Parámetros de la lognormal: elegimos una desviación logarítmica moderada
sigma_ln = 0.6
mu_ln = {k: np.log(v) - 0.5 * sigma_ln**2 for k, v in sev_media.items()}

# Definición de segmentos y multiplicadores de frecuencia
segmentos = pd.DataFrame({
    'segmento': ['Abogados', 'Administradores', 'Agentes', 'Financieros', 'Ingenieros', 'Médicos', 'Odontólogos', 'Oficinas Privadas',
                 'Profesores', 'Servidores Públicos', 'Veterinarios'],
    'prob_segmento': [0.005, 0.011, 0.242, 0.009, 0.055, 0.317, 0.008, 0.230, 0.007, 0.006, 0.11],
    'mult_freq': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
})
segmentos

,segmento,prob_segmento,mult_freq
0,Abogados,0.005,1.0
1,Administradores,0.011,1.0
2,Agentes,0.242,1.0
3,Financieros,0.009,1.0
4,Ingenieros,0.055,1.0
5,Médicos,0.317,1.0
6,Odontólogos,0.008,1.0
7,Oficinas Privadas,0.230,1.0
8,Profesores,0.007,1.0
9,Servidores Públicos,0.006,1.0


### Simulación de pólizas y siniestros

In [3]:
# Asignamos segmento a cada póliza
polizas = pd.DataFrame({
    'id_poliza': np.arange(1, n_polizas+1),
    'segmento': np.random.choice(segmentos['segmento'], size=n_polizas, p=segmentos['prob_segmento'])
})

# Añadimos multiplicador de frecuencia
polizas = polizas.merge(segmentos[['segmento', 'mult_freq']], on='segmento', how='left')

# Función para simular siniestros y severidades por cobertura
def simular_cobertura(freq_base, mult_freq, mu, sigma, size):
    lam = freq_base * mult_freq
    # Número de siniestros ~ Poisson(lam)
    n_siniestros = np.random.poisson(lam, size=size)
    severidades = []
    for n in n_siniestros:
        if n == 0:
            severidades.append(0.0)
        else:
            # Suma de siniestros lognormales
            severidades.append(np.sum(np.random.lognormal(mean=mu, sigma=sigma, size=n)))
    return np.array(n_siniestros), np.array(severidades)

# Simulamos para cada cobertura
n_siniestros_AeI, siniestro_AeI = simular_cobertura(freq_base['AeI'], polizas['mult_freq'], mu_ln['AeI'], sigma_ln, n_polizas)
n_siniestros_DO, siniestro_DO = simular_cobertura(freq_base['DO'], polizas['mult_freq'], mu_ln['DO'], sigma_ln, n_polizas)
n_siniestros_EO, siniestro_EO = simular_cobertura(freq_base['EO'], polizas['mult_freq'], mu_ln['EO'], sigma_ln, n_polizas)
n_siniestros_RChosp, siniestro_RChosp = simular_cobertura(freq_base['RChosp'], polizas['mult_freq'], mu_ln['RChosp'], sigma_ln, n_polizas)
n_siniestros_RCprof, siniestro_RCprof = simular_cobertura(freq_base['RCprof'], polizas['mult_freq'], mu_ln['RCprof'], sigma_ln, n_polizas)

# Incorporamos a la cartera
cartera = polizas.copy()
cartera['n_siniestros_AeI'] = n_siniestros_AeI
cartera['siniestro_AeI'] = siniestro_AeI
cartera['n_siniestros_DO'] = n_siniestros_DO
cartera['siniestro_DO'] = siniestro_DO
cartera['n_siniestros_EO'] = n_siniestros_EO
cartera['siniestro_EO'] = siniestro_EO
cartera['n_siniestros_RChosp'] = n_siniestros_RChosp
cartera['siniestro_RChosp'] = siniestro_RChosp
cartera['n_siniestros_RCprof'] = n_siniestros_RCprof
cartera['siniestro_RCprof'] = siniestro_RCprof
cartera['siniestro_total'] = cartera[['siniestro_AeI','siniestro_DO','siniestro_EO','siniestro_RChosp','siniestro_RCprof']].sum(axis=1)
cartera

,id_poliza,segmento,mult_freq,n_siniestros_AeI,siniestro_AeI,n_siniestros_DO,siniestro_DO,n_siniestros_EO,siniestro_EO,n_siniestros_RChosp,siniestro_RChosp,n_siniestros_RCprof,siniestro_RCprof,siniestro_total
0,1,Agentes,1.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0.0
1,2,Servidores Públicos,1.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0.0
2,3,Veterinarios,1.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0.0
3,4,Médicos,1.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0.0
4,5,Médicos,1.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199714,199715,Médicos,1.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0.0
199715,199716,Veterinarios,1.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0.0
199716,199717,Agentes,1.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0.0
199717,199718,Médicos,1.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0.0


### Cálculo de primas y razón de siniestralidad

In [4]:
pd.options.display.float_format = '{:,.2f}'.format
# Prima pura por cobertura (frecuencia x severidad media) por póliza
prima_AeI_pura = freq_base['AeI']   * cartera['mult_freq'] * sev_media['AeI']
prima_DO_pura = freq_base['DO'] * cartera['mult_freq'] * sev_media['DO']
prima_EO_pura = freq_base['EO']   * cartera['mult_freq'] * sev_media['EO']
prima_RChosp_pura = freq_base['RChosp'] * cartera['mult_freq'] * sev_media['RChosp']
prima_RCprof_pura = freq_base['RCprof']   * cartera['mult_freq'] * sev_media['RCprof']

# Aplicamos margen de gastos (40%)
margin = 0.4
cartera['prima_AeI_comercial'] = prima_AeI_pura * (1 + margin)
cartera['prima_DO_comercial'] = prima_DO_pura * (1 + margin)
cartera['prima_EO_comercial'] = prima_EO_pura * (1 + margin)
cartera['prima_RChosp_comercial'] = prima_RChosp_pura * (1 + margin)
cartera['prima_RCprof_comercial'] = prima_RCprof_pura * (1 + margin)

cartera['prima_total'] = cartera[['prima_AeI_comercial', 'prima_DO_comercial', 'prima_EO_comercial', 'prima_RChosp_comercial',
                                  'prima_RCprof_comercial']].sum(axis=1)

# Razón de siniestralidad (loss ratio) por póliza
cartera['LR'] = np.where(cartera['prima_total'] > 0, cartera['siniestro_total'] / cartera['prima_total'], np.nan)

# Resumen por segmento
resumen = cartera.groupby('segmento').agg({
    'id_poliza': 'count',
    'siniestro_total': 'sum',
    'prima_total': 'sum'
})
resumen.rename(columns={'id_poliza': 'polizas'}, inplace=True)
resumen['LR_promedio'] = resumen['siniestro_total'] / resumen['prima_total']
resumen

,polizas,siniestro_total,prima_total,LR_promedio
segmento,,,,
Abogados,970,"443,689.88","1,104,181.18",0.40
Administradores,2244,"2,082,475.77","2,554,415.02",0.82
Agentes,48252,"39,623,507.05","54,926,752.81",0.72
Financieros,1819,"1,391,360.40","2,070,624.29",0.67
Ingenieros,11133,"9,345,387.24","12,673,040.27",0.74
Médicos,63238,"50,100,621.05","71,985,782.86",0.70
Odontólogos,1599,"910,746.57","1,820,191.45",0.50
Oficinas Privadas,45719,"36,213,234.04","52,043,360.11",0.70
Profesores,1388,"798,182.10","1,580,003.58",0.51


In [5]:
siniestro_total_cartera = cartera['siniestro_total'].sum()
prima_total_cartera = cartera['prima_total'].sum()
LR_total = siniestro_total_cartera / prima_total_cartera
LR_total

np.float64(0.6982860248595213)

In [6]:
# Guardamos la cartera simulada en un archivo CSV
cartera.to_csv('cartera_ResponsabilidadCivil_simulada_Python.csv', index=False)
cartera.head()

,id_poliza,segmento,mult_freq,n_siniestros_AeI,siniestro_AeI,n_siniestros_DO,siniestro_DO,n_siniestros_EO,siniestro_EO,n_siniestros_RChosp,...,n_siniestros_RCprof,siniestro_RCprof,siniestro_total,prima_AeI_comercial,prima_DO_comercial,prima_EO_comercial,prima_RChosp_comercial,prima_RCprof_comercial,prima_total,LR
0,1,Agentes,1.00,0,0.00,0,0.00,0,0.00,0,...,0,0.00,0.00,23.50,0.04,1.44,4.80,"1,108.55","1,138.33",0.00
1,2,Servidores Públicos,1.00,0,0.00,0,0.00,0,0.00,0,...,0,0.00,0.00,23.50,0.04,1.44,4.80,"1,108.55","1,138.33",0.00
2,3,Veterinarios,1.00,0,0.00,0,0.00,0,0.00,0,...,0,0.00,0.00,23.50,0.04,1.44,4.80,"1,108.55","1,138.33",0.00
3,4,Médicos,1.00,0,0.00,0,0.00,0,0.00,0,...,0,0.00,0.00,23.50,0.04,1.44,4.80,"1,108.55","1,138.33",0.00
4,5,Médicos,1.00,0,0.00,0,0.00,0,0.00,0,...,0,0.00,0.00,23.50,0.04,1.44,4.80,"1,108.55","1,138.33",0.00


In [7]:
monto_total_siniestros = cartera['siniestro_total'].sum()
monto_total_siniestros

np.float64(158752779.95018584)

In [8]:
total_prima = cartera['prima_total'].sum()
total_prima

np.float64(227346351.34953925)

In [9]:
cartera['LR_2'] = cartera['siniestro_total'] / cartera['prima_total']
lr_agregado = cartera['siniestro_total'].sum() / cartera['prima_total'].sum()
print(lr_agregado)

0.6982860248595213


In [10]:
LR_promedio = cartera['LR'].mean()
LR_promedio

np.float64(0.6982860248595213)

In [11]:
LR_mediana = cartera['LR'].median()
LR_mediana

0.0